[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/43_rollout_solution.ipynb)

#  Solution: Trajectory Rollout

Reference solution for environment rollouts  the core data-collection primitive in RL.

```text
1. state = env.reset()
2. action = policy_fn(state)
3. next_state, reward, done = env.step(action)
4. Record (state, action, reward, next_state, done)
5. Repeat until done or max_steps
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  SOLUTION

def rollout(policy_fn, env, max_steps=100):
    """Collect trajectory from policy-environment interaction.
    
    Follows the standard RL loop used in Gym/Gymnasium, stable-baselines3, etc.
    Each transition is a dict to be flexible with heterogeneous state/action types.
    """
    state = env.reset()
    trajectory = []
    
    for _ in range(max_steps):
        action = policy_fn(state)
        next_state, reward, done = env.step(action)
        
        trajectory.append({
            "state": state.item() if isinstance(state, torch.Tensor) else state,
            "action": action.item() if isinstance(action, torch.Tensor) else action,
            "reward": reward,
            "next_state": next_state.item() if isinstance(next_state, torch.Tensor) else next_state,
            "done": done,
        })
        
        if done:
            break
        state = next_state
    
    return trajectory

In [ ]:
#  Verify

class SimpleEnv:
    def reset(self): self.c = 0; return torch.tensor(0.0)
    def step(self, a):
        self.c += 1
        return torch.tensor(float(self.c)), float(self.c), self.c >= 3

traj = rollout(lambda s: torch.tensor(1.0), SimpleEnv())
print(f"{len(traj)} steps collected")
for t in traj:
    print(f"  state={t['state']}, action={t['action']}, reward={t['reward']}, done={t['done']}")

In [ ]:
# Run judge
from torch_judge import check
check("rollout")